In [3]:
# ============================================================
# CELL 1 — Import Libraries
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, RocCurveDisplay)

import warnings
warnings.filterwarnings('ignore')

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

In [6]:
# ============================================================
# CELL 5 — Final Scaled Dataset for Modeling
# ============================================================
# Keep only relevant columns for ML (drop UDI, Product ID, original Type)
model_cols = ['Type_Encoded'] +  ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df_final = [model_cols]

df_final.head()

AttributeError: 'list' object has no attribute 'head'

In [ ]:
# ============================================================
# CELL 6 — Verify Scaling Correctness
# ============================================================
print("Mean of scaled columns (should be ~0):")
print(df_scaled[num_cols].mean())

print("\nStandard Deviation of scaled columns (should be ~1):")
print(df_scaled[num_cols].std())

In [ ]:
# ============================================================
# CELL 3 — Dataset Overview: Shape & Columns
# ============================================================
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])
print("\nColumn Names:\n", list(df.columns))

In [ ]:
# ============================================================
# CELL 4 — Column Descriptions
# ============================================================
column_descriptions = {
    "UDI": "Unique row identifier (index number).",
    "Product ID": "Unique product identifier with a letter prefix (L/M/H) indicating quality variant.",
    "Type": "Product quality type: L (Low), M (Medium), H (High).",
    "Air temperature [K]": "Ambient air temperature around the machine, in Kelvin.",
    "Process temperature [K]": "Internal process/machine temperature, in Kelvin.",
    "Rotational speed [rpm]": "Speed of the machine's spindle/rotor in rotations per minute.",
    "Torque [Nm]": "Rotational force applied by the machine, in Newton-metres.",
    "Tool wear [min]": "Total minutes the cutting tool has been in use (wear accumulation).",
    "Machine failure": "Binary target: 1 if the machine failed (any failure mode), else 0.",
    "TWF": "Tool Wear Failure indicator (1 = failure due to worn-out tool).",
    "HDF": "Heat Dissipation Failure indicator (1 = failure due to poor heat dissipation).",
    "PWF": "Power Failure indicator (1 = failure due to power issues, over/under power).",
    "OSF": "Overstrain Failure indicator (1 = failure due to overstrain on the tool/machine).",
    "RNF": "Random Failure indicator (1 = failure with no identifiable specific cause)."
}
for col, desc in column_descriptions.items():
    if col in df.columns:
        print(f"{col}: {desc}")

In [ ]:
# ============================================================
# CELL 5 — Data Types & Feature Classification
# ============================================================
print("Data Types:\n")
print(df.dtypes)

numerical_features = df.select_dtypes(include=np.number).columns.tolist()
categorical_features = df.select_dtypes(include='object').columns.tolist()

print("\nNumerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

In [ ]:
# ============================================================
# CELL 6 — Missing Values Check
# ============================================================
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# ============================================================
# CELL 7 — Duplicate Rows Check
# ============================================================
duplicate_count = df.duplicated().sum()
print("Number of Duplicate Rows:", duplicate_count)

id_cols = [c for c in ['UDI', 'Product ID'] if c in df.columns]
if id_cols:
    print("Duplicate Product IDs:", df.duplicated(subset=id_cols).sum())

In [ ]:
# ============================================================
# CELL 8 — Invalid / Inconsistent Value Detection
# ============================================================
num_cols_check = [c for c in ['Air temperature [K]', 'Process temperature [K]',
                               'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
                   if c in df.columns]

for col in num_cols_check:
    negative_count = (df[col] < 0).sum()
    print(f"{col}: Negative values = {negative_count}, Min = {df[col].min()}, Max = {df[col].max()}")

# Check Process temperature should generally be >= Air temperature
if 'Process temperature [K]' in df.columns and 'Air temperature [K]' in df.columns:
    inconsistent = (df['Process temperature [K]'] < df['Air temperature [K]']).sum()
    print("\nRows where Process temperature < Air temperature:", inconsistent)

# Check target consistency: Machine failure should be 1 if any failure mode is 1
failure_modes = [c for c in ['TWF', 'HDF', 'PWF', 'OSF', 'RNF'] if c in df.columns]
if failure_modes and 'Machine failure' in df.columns:
    any_failure = df[failure_modes].sum(axis=1) > 0
    mismatch = (df['Machine failure'] == 0) & any_failure
    print("Rows with a failure mode = 1 but Machine failure = 0:", mismatch.sum())

In [ ]:
# ============================================================
# CELL 9 — Data Cleaning (Drop irrelevant ID columns for modeling later)
# ============================================================
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()

# Drop non-informative identifier columns (kept aside, not used in ML)
id_cols_to_drop = [c for c in ['UDI', 'Product ID'] if c in df_clean.columns]
df_model = df_clean.drop(columns=id_cols_to_drop)

print("Shape after cleaning:", df_clean.shape)
print("Shape for modeling (IDs dropped):", df_model.shape)

In [ ]:
# ============================================================
# CELL 10 — Descriptive Statistics
# ============================================================
desc_stats = df_model.describe().T
desc_stats['median'] = df_model.select_dtypes(include=np.number).median()
desc_stats['variance'] = df_model.select_dtypes(include=np.number).var()
desc_stats['mode'] = df_model.select_dtypes(include=np.number).mode().iloc[0]
desc_stats

In [ ]:
# ============================================================
# CELL 11 — EDA: Histograms
# ============================================================
num_cols = ['Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
num_cols = [c for c in num_cols if c in df_model.columns]

df_model[num_cols].hist(bins=30, figsize=(14, 10), edgecolor='black')
plt.suptitle("Histograms of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 12 — EDA: Boxplots
# ============================================================
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 5))
for i, col in enumerate(num_cols):
    sns.boxplot(y=df_model[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 13 — EDA: Scatter Plots
# ============================================================
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_model, x='Torque [Nm]', y='Rotational speed [rpm]',
                 hue='Machine failure', palette='coolwarm', alpha=0.6)
plt.title("Torque vs Rotational Speed (colored by Machine Failure)")
plt.show()

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_model, x='Air temperature [K]', y='Process temperature [K]',
                 hue='Machine failure', palette='coolwarm', alpha=0.6)
plt.title("Air Temp vs Process Temp (colored by Machine Failure)")
plt.show()

In [ ]:
# ============================================================
# CELL 14 — EDA: Pair Plot
# ============================================================
sns.pairplot(df_model[num_cols + ['Machine failure']], hue='Machine failure',
             palette='coolwarm', corner=True)
plt.suptitle("Pair Plot of Numerical Features", y=1.02)
plt.show()

In [ ]:
# ============================================================
# CELL 15 — EDA: Correlation Heatmap
# ============================================================
plt.figure(figsize=(12, 8))
corr_matrix = df_model.select_dtypes(include=np.number).corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ============================================================
# CELL 16 — EDA: Count Plots
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
targets = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
targets = [t for t in targets if t in df_model.columns]

for i, col in enumerate(targets):
    r, c = divmod(i, 3)
    sns.countplot(x=df_model[col], ax=axes[r][c], palette='Set2')
    axes[r][c].set_title(f"Count Plot: {col}")
plt.tight_layout()
plt.show()

if 'Type' in df_model.columns:
    plt.figure(figsize=(6, 4))
    sns.countplot(x=df_model['Type'], palette='Set3')
    plt.title("Count Plot: Product Type")
    plt.show()

In [ ]:
# ============================================================
# CELL 17 — EDA: Distribution Plots (KDE)
# ============================================================
fig, axes = plt.subplots(1, len(num_cols), figsize=(22, 5))
for i, col in enumerate(num_cols):
    sns.histplot(df_model[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f"Distribution: {col}")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 18 — EDA: Violin Plots
# ============================================================
fig, axes = plt.subplots(1, len(num_cols), figsize=(22, 5))
for i, col in enumerate(num_cols):
    sns.violinplot(x=df_model['Machine failure'], y=df_model[col], ax=axes[i], palette='muted')
    axes[i].set_title(f"{col} vs Machine Failure")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 19 — EDA: Pie Charts
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

df_model['Machine failure'].value_counts().plot.pie(
    autopct='%1.1f%%', labels=['No Failure', 'Failure'],
    colors=['#8fd9a8', '#f28b82'], ax=axes[0])
axes[0].set_title("Machine Failure Distribution")
axes[0].set_ylabel('')

if 'Type' in df_model.columns:
    df_model['Type'].value_counts().plot.pie(
        autopct='%1.1f%%', colors=['#a8d0e6', '#f6c85f', '#6cc551'], ax=axes[1])
    axes[1].set_title("Product Type Distribution")
    axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 20 — Outlier Detection: IQR Method
# ============================================================
outlier_summary_iqr = {}
for col in num_cols:
    Q1 = df_model[col].quantile(0.25)
    Q3 = df_model[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df_model[(df_model[col] < lower) | (df_model[col] > upper)]
    outlier_summary_iqr[col] = len(outliers)

pd.DataFrame.from_dict(outlier_summary_iqr, orient='index', columns=['Outlier Count (IQR)'])

In [ ]:
# ============================================================
# CELL 21 — Outlier Detection: Z-Score Method
# ============================================================
outlier_summary_z = {}
for col in num_cols:
    z_scores = np.abs(stats.zscore(df_model[col]))
    outlier_summary_z[col] = (z_scores > 3).sum()

pd.DataFrame.from_dict(outlier_summary_z, orient='index', columns=['Outlier Count (Z-score > 3)'])

In [ ]:
# ============================================================
# CELL 22 — Correlation Analysis: Top Correlated Pairs
# ============================================================
corr_pairs = corr_matrix.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs != 1.0]
corr_pairs = corr_pairs[~corr_pairs.index.duplicated(keep='first')]
print("Top Positive/Negative Correlations:")
print(corr_pairs.drop_duplicates().head(10))
print("...")
print(corr_pairs.drop_duplicates().tail(10))

In [ ]:
# ============================================================
# CELL 24 — Target Variable Analysis: Class Distribution
# ============================================================
target_cols = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
target_cols = [t for t in target_cols if t in df_fe.columns]

for col in target_cols:
    counts = df_fe[col].value_counts()
    pct = df_fe[col].value_counts(normalize=True) * 100
    print(f"\n{col} Distribution:")
    print(pd.DataFrame({'Count': counts, 'Percentage': pct.round(2)}))

In [ ]:
# ============================================================
# CELL 25 — ML Preparation: Feature Selection & Scaling
# ============================================================
feature_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]',
                 'Torque [Nm]', 'Tool wear [min]', 'Type_Encoded',
                 'Temp_Difference', 'Power_W', 'Wear_Torque_Ratio', 'Strain_Index']

X = df_fe[feature_cols]
y = df_fe['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# ============================================================
# CELL 26 — Model Building: Train All Models
# ============================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(eval_metric='logloss', random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB()
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully.")

In [ ]:
# ============================================================
# CELL 27 — Model Evaluation: Metrics for Each Model
# ============================================================
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)

    results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1-Score": f1, "ROC-AUC": roc_auc
    })

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
results_df

In [ ]:
# ============================================================
# CELL 28 — Confusion Matrices for Each Model
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(name)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

for j in range(len(trained_models), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 29 — ROC Curve Comparison
# ============================================================
plt.figure(figsize=(10, 8))
for name, model in trained_models.items():
    RocCurveDisplay.from_estimator(model, X_test_scaled, y_test, name=name, ax=plt.gca())
plt.title("ROC Curve Comparison - All Models")
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.show()

In [ ]:
# ============================================================
# CELL 30 — Feature Importance (Random Forest & XGBoost)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_importance = pd.Series(trained_models["Random Forest"].feature_importances_,
                           index=feature_cols).sort_values(ascending=False)
sns.barplot(x=rf_importance.values, y=rf_importance.index, ax=axes[0], palette='viridis')
axes[0].set_title("Random Forest Feature Importance")

xgb_importance = pd.Series(trained_models["XGBoost"].feature_importances_,
                            index=feature_cols).sort_values(ascending=False)
sns.barplot(x=xgb_importance.values, y=xgb_importance.index, ax=axes[1], palette='magma')
axes[1].set_title("XGBoost Feature Importance")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 31 — Classification Reports (Detailed)
# ============================================================
for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# ============================================================
# CELL 32 — Best Model Summary
# ============================================================
best_model_row = results_df.iloc[0]
print("Best Performing Model (by ROC-AUC):")
print(best_model_row)

In [ ]:
# ============================================================
# CELL 5 — Final Scaled Dataset for Modeling
# ============================================================
# Keep only relevant columns for ML (drop UDI, Product ID, original Type)
model_cols = ['Type_Encoded'] + num_cols + ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df_final = df_scaled[model_cols]

df_final.head()

In [ ]:
# ============================================================
# CELL 6 — Verify Scaling Correctness
# ============================================================
print("Mean of scaled columns (should be ~0):")
print(df_scaled[num_cols].mean())

print("\nStandard Deviation of scaled columns (should be ~1):")
print(df_scaled[num_cols].std())

In [ ]:

# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ============================================================
# CELL 2 — Load Data
# ============================================================
df = pd.read_csv(r"C:\Users\admin\Downloads\Maintenance Dataset (AI4I 2020) (2).csv")   # <-- apni actual CSV file ka path yaha daalo
df.head()

# ============================================================
# CELL 3 — Encoding (Product ID & Type)
# ============================================================
df_encoded = df.copy()

le_product = LabelEncoder()
df_encoded['Product_ID_Encoded'] = le_product.fit_transform(df_encoded['Product ID'])

le_type = LabelEncoder()
df_encoded['Type_Encoded'] = le_type.fit_transform(df_encoded['Type'])

# ============================================================
# CELL 4 — Final Dataframe with Numeric Type & Product ID
# ============================================================
df_numeric = df_encoded.drop(columns=['Product ID', 'Type'])

df_numeric = df_numeric[['UDI', 'Product_ID_Encoded', 'Type_Encoded',
                          'Air temperature [K]', 'Process temperature [K]',
                          'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
                          'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']]

df_numeric.head()

# ============================================================
# CELL 5 — Scaling
# ============================================================
features_to_scale = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

scaler = StandardScaler()
df_scaled = df_numeric.copy()
df_scaled[features_to_scale] = scaler.fit_transform(df_numeric[features_to_scale])

df_scaled.head()

# ============================================================
# CELL 15 — Prepare df_model
# ============================================================
df_model = df_numeric.copy()   # ya df_scaled.copy() agar scaled version aage use karna hai

df_model.head()

In [ ]:
LogisticRegression(max_iter=5000, solver='saga', random_state=42)

In [ ]:
# ============================================================
# CELL 20 (Updated) — Train-Test Split + Scaling (Fixed)
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# UDI aur Product_ID_Encoded model ke liye useful nahi (ye sirf identifiers hain)
# Inhe drop karna better practice hai
X = df_numeric.drop(columns=['UDI', 'Product_ID_Encoded',
                              'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df_numeric['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Ab saare remaining numeric features scale karo (Type_Encoded ko chhod ke, wo categorical hai)
features_to_scale = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
X_test_scaled[features_to_scale] = scaler.transform(X_test[features_to_scale])

In [ ]:
# ============================================================
# CELL 20 — Train-Test Split + Scaling
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_numeric.drop(columns=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'])
y = df_numeric['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

features_to_scale = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]'
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[features_to_scale] = scaler.fit_transform(X_train[features_to_scale])
X_test_scaled[features_to_scale] = scaler.transform(X_test[features_to_scale])

# ============================================================
# CELL 21 — Train Multiple Models
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

trained_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True, random_state=42)
}

for name, model in trained_models.items():
    model.fit(X_train_scaled, y_train)
    print(f"{name} trained successfully.")

In [ ]:
# ============================================================
# CELL 22 — Evaluate All Models + Comparison Chart
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)

results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc
    })

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

print("Model Comparison Table:")
display(results_df)

# ------------------------------------------------------------
# Grouped Bar Chart — sab models, sab metrics ek